# KLT Analysis on Real Data: The Voyager 1 Case Study
This notebook focuses on applying the Karhunen-Loève Transform (KLT) to telemetry data from the Voyager 1 spacecraft. 

Unlike ideal synthetic signals, real observatory data (such as GUPPI RAW files from the Green Bank Telescope) contains complex noise structures, potential spectral inversions, and hardware-specific artifacts (e.g., I/Q swaps or NPOL packing variations). 

Here, we utilize the multi-component capabilities of the KLT to isolate the spacecraft's carrier and sidebands from the background noise floor.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from seti_klt.KLT import KLT

# Import our unified I/O, logging, and directory tools
from seti_klt.utils.io import (
    SimpleLogger, 
    LogLevel, 
    get_data_path, 
    prepare_output_dir, 
    save_figure
)

np.random.seed(42)

# Initialize the logging environment
logger = SimpleLogger(level=LogLevel.INFO, filename="voyager1_klt_pipeline.log")
logger.section("Voyager 1 KLT Pipeline Initialization")

/mnt/c/Users/gigig/Documents/University/SETI/codes/.venv/lib/python3.12/site-packages/blimpy/__init__.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


Logging to: /mnt/c/Users/gigig/Documents/University/SETI/codes/notebooks/Voyager1_Signal/outputs/logs/voyager1_klt_pipeline.log

=== VOYAGER 1 KLT PIPELINE INITIALIZATION


## 1. Parameters and Data Path Configuration
We set up the file parameters and processing dimensions. Input files are read from the central `data/` directory, while all generated products (logs, figures, and extracted data) will be saved inside a local `outputs/` folder created on-the-fly where this notebook runs.

In [2]:
# Processing dimensions
WINDOW_SIZE  = 1024
NUM_SAMPLES  = 2**24  # Full full-rank covariance matrix analysis (K >= W)
N_EIGENVECS  = 3      # Number of dominant eigenvectors to retain for reconstruction

POLARIZATION = 0

# Locate input file dynamically from the global data folder
data_root = get_data_path()
FILE_NAME = "blc5_2bit_guppi_57396_VOYAGER1_0002"
FILE_STEM = str(data_root / FILE_NAME)
FILE_PATH = str(f"{FILE_STEM}.0000.raw")

# Setup a local directory for raw validation or sub-selections if needed
local_raw_dir = prepare_output_dir("outputs/raw_data")

logger.summary_stats({
    "Input File": FILE_NAME,
    "Polarization": POLARIZATION,
    "Window Size (W)": WINDOW_SIZE,
    "Samples Loaded (N)": NUM_SAMPLES,
    "Retained Components": N_EIGENVECS
}, title="Voyager 1 Analysis Setup")


=== VOYAGER 1 ANALYSIS SETUP
[11:56:50] [INFO]     Input File: blc5_2bit_guppi_57396_VOYAGER1_0002
[11:56:50] [INFO]     Polarization: 0
[11:56:50] [INFO]     Window Size (W): 1024
[11:56:50] [INFO]     Samples Loaded (N): 16777216
[11:56:50] [INFO]     Retained Components: 3


## 2. Data Loading and Covariance Analysis
We initialize the KLT core class, pull the raw voltage samples from the GUPPI block, and perform the Eigendecomposition. 

*Note on Hardware Inversions:* Real GBT data may require spectral flip corrections or complex conjugation (`Q = -Q`) due to intermediate frequency downconversion stages. The `KLT` loading pipeline handles these matrix constraints natively.

In [3]:
klt = KLT()

# Load the raw complex voltage samples
klt.load_data_from_guppi(
    FILE_PATH,
    FILE_STEM,
    channel=0,
    num_samples=NUM_SAMPLES,
    polarization=POLARIZATION
)

# Run the Cross-Check Channel Power Scan to verify signal presence
fig_power, _, channel = klt.channel_power_scan(
    FILE_PATH,
    polarization=POLARIZATION
)

save_figure(fig_power, filename=f"{FILE_STEM}_channel_power_scan.png")


=== LOADING GUPPI DATA
[11:56:53] [INFO]     File: '/mnt/c/Users/gigig/Documents/University/SETI/codes/data/blc5_2bit_guppi_57396_VOYAGER1_0002.0000.raw' | channel: 0 | polarization: 0 | requested samples: 16,777,216.


Reading blocks: 17.0Msamp [01:26, 198ksamp/s]                         


[11:58:19] [INFO]     Loaded 16,777,216 samples (requested: 16,777,216, blocks read: 33).
[11:58:19] [INFO]     File polarizations (NPOL): 4.

=== CHANNEL POWER SCAN
[11:58:20] [INFO]     not specified.
[11:58:25] [INFO]     Dominant channel: 44 (centre: 8755.3711 MHz, peak power: 3.046e+11).
Figure saved → /mnt/c/Users/gigig/Documents/University/SETI/codes/notebooks/Voyager1_Signal/outputs/figures/blc5_2bit_guppi_57396_VOYAGER1_0002_channel_power_scan.png


PosixPath('/mnt/c/Users/gigig/Documents/University/SETI/codes/notebooks/Voyager1_Signal/outputs/figures/blc5_2bit_guppi_57396_VOYAGER1_0002_channel_power_scan.png')

## 3. Multi-Component CKLT Execution
We apply the Covariance KLT (`apply_cklt`). Since the Voyager 1 signal includes modulated telemetry sidebands, a single eigenvector might not capture the full information. We track the top `N_EIGENVECS` to observe energy distribution across the signal subspace.

In [4]:
# Load the raw complex voltage samples
klt.load_data_from_guppi(
    FILE_PATH,
    FILE_STEM,
    channel=channel,
    num_samples=NUM_SAMPLES,
    polarization=POLARIZATION
)

klt.apply_cklt(window_size=WINDOW_SIZE, n_eigenvectors=N_EIGENVECS)

# Extract and log eigenvalue metrics
eigvals = klt.get_eigenvalues()
gap_ratio = eigvals[0] / eigvals[1]
logger.info(f"Top 3 Eigenvalues: {eigvals[0]:.2f}, {eigvals[1]:.2f}, {eigvals[2]:.2f}")
logger.info(f"First-to-Second Eigenvalue Gap Ratio: {gap_ratio:.2f}x")


=== LOADING GUPPI DATA
[11:58:31] [INFO]     File: '/mnt/c/Users/gigig/Documents/University/SETI/codes/data/blc5_2bit_guppi_57396_VOYAGER1_0002.0000.raw' | channel: 44 | polarization: 0 | requested samples: 16,777,216.


Reading blocks: 17.0Msamp [01:21, 208ksamp/s]                         


[11:59:53] [INFO]     Loaded 16,777,216 samples (requested: 16,777,216, blocks read: 33).
[11:59:53] [INFO]     File polarizations (NPOL): 4.

=== APPLYING C-KLT
[11:59:54] [INFO]     Window size: 1024 samples | windows (K): 16384 | eigenvectors to keep: 3.
[11:59:55] [INFO]     Top eigenvalue: 4086.5046 | 2nd eigenvalue: 2821.3508 | gap ratio λ₁/λ₂: 1.45x.
[11:59:55] [INFO]     Reconstruction complete — output length: 16,777,216 samples.
[11:59:55] [INFO]     Top 3 Eigenvalues: 4086.50, 2821.35, 2736.32
[11:59:55] [INFO]     First-to-Second Eigenvalue Gap Ratio: 1.45x


## 4. Performance Metrics and Visualization
We evaluate the reconstruction quality by plotting the eigenspectrum, the dynamic waterfall comparison, and the integrated Power Spectral Density (PSD). The `save_figure` tool automatically roots these plots under `outputs/figures/` locally.

In [5]:
logger.section("Generating Reference-Style Diagnostic Plots")

# 1. Plot Eigenspectrum to inspect the rank and the noise tail drop-off
fig_eig, _ = klt.plot_eigenspectrum(n_components=N_EIGENVECS)
save_figure(fig_eig, filename=f"{FILE_STEM}_klt_eigenspectrum.png")

# 2. Plot Spectrogram/Waterfall comparison (Noisy Input vs KLT Filtered Subspace)
fig_wf, _ = klt.plot_waterfall_comparison(channel=channel)
save_figure(fig_wf, filename=f"{FILE_STEM}_klt_waterfall_comparison.png")

# 3. Plot Integrated PSD Comparison using Welch estimation
fig_psd, _, peak = klt.plot_psd_comparison(channel=channel)
save_figure(fig_psd, filename=f"{FILE_STEM}_klt_psd_comparison.png")

logger.info("Pipeline executed successfully. All assets saved locally.")
logger.close()


=== GENERATING REFERENCE-STYLE DIAGNOSTIC PLOTS
Figure saved → /mnt/c/Users/gigig/Documents/University/SETI/codes/notebooks/Voyager1_Signal/outputs/figures/blc5_2bit_guppi_57396_VOYAGER1_0002_klt_eigenspectrum.png
Figure saved → /mnt/c/Users/gigig/Documents/University/SETI/codes/notebooks/Voyager1_Signal/outputs/figures/blc5_2bit_guppi_57396_VOYAGER1_0002_klt_waterfall_comparison.png
[12:00:09] [INFO]     PSD comparison — peak noisy: 8755.3711 MHz, peak reconstructed: 8755.3711 MHz.
Figure saved → /mnt/c/Users/gigig/Documents/University/SETI/codes/notebooks/Voyager1_Signal/outputs/figures/blc5_2bit_guppi_57396_VOYAGER1_0002_klt_psd_comparison.png
[12:00:10] [INFO]     Pipeline executed successfully. All assets saved locally.
Log closed: /mnt/c/Users/gigig/Documents/University/SETI/codes/notebooks/Voyager1_Signal/outputs/logs/voyager1_klt_pipeline.log
